<a href="https://colab.research.google.com/github/tomonari-masada/course2026-sml/blob/main/11_document_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# クラスタリング
* クラスタリングの代表的な手法であるk平均法を使ってみる。
* ついでに、言語モデルを使ったテキストマイニングを体験してみる。

## 例題: 文書クラスタリング

* Transformerベースの日本語対応言語モデルを使って、テキストのベクトル表現を得る。
  * Transformerというニューラルネットワークについては、いずれ学びます。
  * 有名な解説記事 https://jalammar.github.io/illustrated-transformer/
* テキストをベクトルとして表現することを「embedする」と言う。
  * embedすることで得られるベクトルのことを「embedding」と言う。
* そして、テキストのembeddingをk平均法でクラスタリングする。

* ランタイムのタイプをGPUにしておく。

## インストール

* datasetsライブラリのバージョンを古くする。
  * 使用するデータセットの都合。

In [ ]:
!pip install datasets==3.6.0

### spaCyの日本語モデル

* 日本語テキストを形態素解析するために使う。
  * たぶん、セッションの再起動は不要。

In [ ]:
!python -m spacy download ja_core_news_sm

### SentenceTransformersライブラリ
* 言語モデルを使ってテキストを埋め込む際に便利なライブラリ。
  * https://sbert.net/index.html
* Google Colabでは改めてインストールする必要はない。

## インポート

In [ ]:
from tqdm.auto import tqdm
import collections
import numpy as np

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import spacy

from datasets import load_dataset
from transformers import set_seed
from sentence_transformers import SentenceTransformer

# 再現性の確保
set_seed(1234)

## データセット
* livedoorニュースコーパスを使う。
  * これを使うために`datasets`のバージョンを古くした。

In [ ]:
ds = load_dataset(
  "shunk031/livedoor-news-corpus",
  train_ratio=0.8, val_ratio=0.1, test_ratio=0.1,
  random_state=42,
  shuffle=True,
  trust_remote_code=True,
)

num_categories = len(set(ds["train"]["category"]))

category_names = [
  'movie-enter',
  'it-life-hack',
  'kaden-channel',
  'topic-news',
  'livedoor-homme',
  'peachy',
  'sports-watch',
  'dokujo-tsushin',
  'smax',
]

print(f"num_categories: {num_categories}")
print(f"category_names: {category_names}")

In [ ]:
ds["train"][0]

In [ ]:
ds["train"]["title"][0]

In [ ]:
ds["train"]["content"][0]

## テキストの埋め込み

* `codefuse-ai/F2LLM-v2-0.6B`を使う。
  * テキストのembeddingにおいて優れている言語モデル。

* 参考: テキスト埋め込みのleaderboard
  * https://huggingface.co/spaces/mteb/leaderboard

* SentenceTransformerを使ったテキストの埋め込みについては、下のWebページを参照。
  * https://sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html

In [ ]:
model_id = "codefuse-ai/F2LLM-v2-0.6B"
model = SentenceTransformer(model_id)

* 試しに、一つだけ、テキストを埋め込んでみる。

In [ ]:
model.encode(ds["train"][0]["title"])

* ライブドアニュースコーパスの全タイトルを埋め込む。

In [ ]:
embeddings = model.encode(ds["train"]["title"], show_progress_bar=True)

* 埋め込みは普通にNumPyの配列として得られている。

In [ ]:
type(embeddings)

* 全記事内容を埋め込むには以下のようにする。

In [ ]:
#content_embeddings = model.encode(ds["train"]["content"], show_progress_bar=True)

* ただし、どのテキストも先頭から`model.max_seq_length`トークンで切られていることに注意。
  * 長いテキストは、途中までの内容しか埋め込みに反映されない。
  * それでも、分類やクラスタリングがうまくいくことも多い。

In [ ]:
model.max_seq_length

* トークン数の調べ方
  * トークナイザにテキストを分割させる。
  * 分割によって得られたトークンの個数を数える。

In [ ]:
model.preprocess([ds["train"][0]["title"]])

In [ ]:
(model.preprocess([ds["train"][0]["title"]])['input_ids']).shape[1]

* トークン化の結果を、サブワードの列として見ると・・・

In [ ]:
model.tokenizer.convert_ids_to_tokens(model.preprocess([ds["train"][0]["title"]])['input_ids'][0])

* 埋め込みを保存。

In [ ]:
with open('embeddings.npy', 'wb') as f:
  np.save(f, embeddings)

In [ ]:
#with open('content_embeddings.npy', 'wb') as f:
#  np.save(f, content_embeddings)

* 読み込みは以下のようにする。

In [ ]:
with open('embeddings.npy', 'rb') as f:
  embeddings = np.load(f)

In [ ]:
#with open('content_embeddings.npy', 'rb') as f:
#  content_embeddings = np.load(f)

## クラスタのラベリングに使う単語の抽出

* 全テキストを形態素解析する。
  * 形態素解析＝単語への分割

In [ ]:
nlp = spacy.load("ja_core_news_sm")
corpus = []
for text in tqdm(ds["train"]["title"]):
  corpus.append(" ".join([token.lemma_ for token in nlp(text)]))

* scikit-learnでTF-IDFを計算する。
* `TfidfVectorizer`の`min_df`パラメータは適当に調節する。
  * クラスタのラベリングに向かないマイナーな単語が含まれないようにする。

In [ ]:
vectorizer = TfidfVectorizer(min_df=20)
X_train = vectorizer.fit_transform(corpus).toarray()
vocab = np.array(vectorizer.get_feature_names_out())

In [ ]:
vocab.size

In [ ]:
print(list(vocab))

## ラベリング用単語の埋め込み

* 各単語について、その単語を含むテキストの埋め込みベクトルの加重平均を求める。
* 加重平均の重みは、各テキストにおけるその単語のTF-IDFの値を使って定める。

In [ ]:
text_weights = X_train / X_train.sum(0)

In [ ]:
vocab_embeddings = np.dot(text_weights.T, embeddings)

## 文書クラスタリング



### k-平均法によるクラスタリング

In [ ]:
n_clusters = 20
kmeans = KMeans(n_clusters=n_clusters, n_init='auto', random_state=123)
kmeans.fit(embeddings)
#kmeans.fit(content_embeddings) # 本文の場合はこちら。
centers = kmeans.cluster_centers_

* クラスタの重心を保存。

In [ ]:
with open(f'centers_{n_clusters}.npy', 'wb') as f:
  np.save(f, centers)

In [ ]:
with open(f'centers_{n_clusters}.npy', 'rb') as f:
  centers = np.load(f)

### クラスタのサイズを調べる

* クラスタのインデックスをキーとし、そのサイズを値とする辞書を作る。

In [ ]:
unique, counts = np.unique(kmeans.labels_, return_counts=True)
size_dict = dict(zip(unique, counts))

* 辞書のエントリを、キーではなく値でソートする。

In [ ]:
print([sorted(size_dict.items(), key=lambda item: item[1], reverse=True)])

## クラスタのラベリング
* 各クラスタの重心に近い単語でラベリングする。

* テキストの埋め込みは、長さ1のベクトルになっている。

In [ ]:
np.linalg.norm(embeddings, axis=-1)

* テキストとラベリング用の単語との類似度はコサイン類似度で測る。

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(vocab_embeddings, centers)

* 重心に近い順に30個の単語を表示する。

In [ ]:
for i in range(similarities.shape[-1]):
  indices = np.argsort(- similarities[:,i])
  print(vocab[indices[:30]])

# 課題
* それぞれのクラスタについて、重心に近い元々のテキスト（つまり記事タイトル）を5件ずつ表示させてみよう。
* それらのテキストの内容に、上で得たラベルが合っているかどうか、確かめよう。